# Análisis Exploratorio de Datos (EDA) + Regresión Lineal  
## Dataset: Wine Quality – Vino Tinto

**Objetivo:**  
Realizar un EDA completo y entrenar modelos de **Regresión Lineal Simple y Múltiple** para predecir la variable objetivo **`quality`** (calidad del vino tinto) a partir de sus características fisicoquímicas.

---
## 🔧 FASE 1: CONFIGURACIÓN E IMPORTACIONES

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

from scipy.stats import shapiro

import warnings
warnings.filterwarnings('ignore')

# Configuración adicional para formato y visualización
pd.options.display.float_format = '{:,.2f}'.format
sns.set_theme(style="whitegrid")

### Análisis FASE 1

En esta celda se importan las librerías necesarias para:

- **Manipulación y análisis de datos:** `pandas`, `numpy`.  
- **Visualización:** `matplotlib`, `seaborn`.  
- **Preprocesamiento y modelado:** `LabelEncoder`, `MinMaxScaler`, `train_test_split`, `LinearRegression`, métricas de evaluación.  
- **Estadística:** `shapiro` para tests de normalidad.  

#### Además, se configura el formato de impresión de números en `pandas` y un estilo visual general para las gráficas con `seaborn`, lo que hace los resultados más legibles y profesionales.

---
## 📂 FASE 2: CARGA E INSPECCIÓN DE DATOS

In [ ]:
# URL del dataset (vino tinto)
datos_vinos = pd.read_csv(
    "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv",
    sep=";"
)

# Crear una copia de trabajo para no alterar los datos originales
datos_trabajo = datos_vinos.copy()

# Mostrar primeras filas
print("=== PRIMERAS 5 FILAS DEL DATASET ===")
display(datos_trabajo.head())

# Mostrar dimensiones del dataset
print(f"\nDimensiones del dataset: {datos_trabajo.shape}")

### Análisis FASE 2

El dataset cargado corresponde a **vino tinto**, con características fisicoquímicas por fila (cada fila es una muestra de vino).

- Dimensiones esperadas del dataset: **1.599 filas** y **12 columnas** (1.599 vinos evaluados y 12 variables).  
- Cada columna representa una propiedad del vino, por ejemplo:
  - `fixed acidity`, `volatile acidity`, `citric acid`, `residual sugar`, etc.  
  - `alcohol`: porcentaje de alcohol del vino.  
  - **`quality`**: es la **variable objetivo**, una calificación sensorial de la calidad del vino (típicamente en escala entera de 0 a 10, usualmente concentrada entre 3 y 8 para este dataset).

##### Esta estructura es adecuada para un problema de **regresión**, donde se busca predecir la calidad (`quality`) a partir de las demás variables numéricas.

---
## 🧹 FASE 3: LIMPIEZA Y TRANSFORMACIÓN DE DATOS
### PASO 3 & 4: Introducción de NaN e Imputación (Limpieza Completa)

In [ ]:
# ── PASO 3: Revisar datos, introducir NaN ──────────────────────────────────────

datos_trabajo.info()
print("\n=== ESTADÍSTICOS DESCRIPTIVOS ===")
display(datos_trabajo.describe())

# Crear valores NaN en 10% de las filas para columnas seleccionadas
porcentaje_nan = 0.10
rng = np.random.default_rng(seed=42)
columnas_nan = ["alcohol", "pH", "chlorides"]

for col in columnas_nan:
    n_filas = int(len(datos_trabajo) * porcentaje_nan)
    indices = rng.choice(datos_trabajo.index, size=n_filas, replace=False)
    datos_trabajo.loc[indices, col] = np.nan

print("\n=== VALORES FALTANTES POR COLUMNA (después de introducir NaN) ===")
display(datos_trabajo.isna().sum())

plt.figure(figsize=(10, 13))
sns.heatmap(datos_trabajo.isna(), cbar=True, cmap='viridis')
plt.title('Mapa de Valores Faltantes')
plt.tight_layout()
plt.show()

# ── PASO 4: Imputación con la mediana ─────────────────────────────────────────

for col in columnas_nan:
    datos_trabajo[col].plot(kind='kde', title=f'Distribución de {col} (antes de imputar)')
    plt.tight_layout()
    plt.show()
    datos_trabajo[col].fillna(datos_trabajo[col].median(), inplace=True)

print("\n=== VALORES FALTANTES TRAS IMPUTACIÓN ===")
display(datos_trabajo.isna().sum())

plt.figure(figsize=(10, 13))
sns.heatmap(datos_trabajo.isna(), cbar=True, cmap='viridis')
plt.title('Mapa de Valores Faltantes (después de imputar)')
plt.tight_layout()
plt.show()

### Análisis FASE 3 – Limpieza y Transformación

**Paso 3 – Introducción de NaN:**  
Se generó intencionalmente un **10% de valores faltantes (NaN)** en las columnas `alcohol`, `pH` y `chlorides`, con el fin de simular condiciones reales de datos incompletos y evaluar la capacidad del EDA para detectarlos. El *heatmap* permite visualizar claramente los puntos donde hay datos faltantes.

**Paso 4 – Imputación con la mediana:**  
Se aplicó una **estrategia de imputación con la mediana** para las tres columnas afectadas. La mediana es una medida robusta ante valores extremos y permite mantener la distribución de los datos sin verse excesivamente influida por outliers. Tras la imputación, el conteo de valores faltantes es cero en todas las columnas, dejando el dataset **listo para el modelado** sin necesidad de eliminar filas completas.

### PASO 5: Conversión de Variables Categóricas a Número

In [ ]:
# Identificar columnas categóricas (tipo 'object')
columnas_categoricas = datos_trabajo.select_dtypes(include=['object']).columns
print(f"Columnas categóricas encontradas: {list(columnas_categoricas)}")

# Aplicar LabelEncoder solo si existen columnas categóricas
le = LabelEncoder()

for col in columnas_categoricas:
    datos_trabajo[col] = le.fit_transform(datos_trabajo[col])
    print(f"Columna '{col}' convertida a numérica mediante LabelEncoder.")

# Verificar conversión
print("\nTipos de datos después de la conversión:")
print(datos_trabajo.dtypes)

En este dataset específico de calidad de vino tinto, no se detectaron columnas de tipo `object`, por lo que **no existen variables categóricas que requieran conversión** mediante `LabelEncoder`. Todas las variables ya están en formato numérico y son directamente utilizables por los algoritmos de machine learning. Aun así, este paso es importante en la plantilla porque en otros datasets con variables categóricas sería obligatorio convertirlas antes de entrenar modelos.

### PASO 6: Normalización

In [ ]:
# Copia antes de normalizar
datos_antes_normalizar = datos_trabajo.copy(deep=True)

# Ver estadísticos antes de normalizar
print("=== ANTES DE NORMALIZACIÓN ===")
print(datos_trabajo.describe().T)

# Aplicar MinMaxScaler
scaler = MinMaxScaler()
columnas_numericas = datos_trabajo.columns  # todas son numéricas
datos_trabajo[columnas_numericas] = scaler.fit_transform(datos_trabajo[columnas_numericas])

# Comparación visual para la variable 'alcohol'
variable = "alcohol"
original_data = datos_antes_normalizar[variable].values.reshape(-1, 1)
normalized_data = datos_trabajo[variable].values.reshape(-1, 1)

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(original_data, ax=ax[0], kde=True, legend=False)
ax[0].set_title(f"Distribución Original de {variable}")
sns.histplot(normalized_data, ax=ax[1], kde=True, legend=False)
ax[1].set_title(f"Distribución Normalizada de {variable}")
plt.tight_layout()
plt.show()

# Ver estadísticos después de normalizar
print("\n=== DESPUÉS DE NORMALIZACIÓN ===")
print(datos_trabajo.describe().T)

La normalización es un paso esencial en el preprocesamiento porque permite llevar todas las variables a la misma escala, evitando que aquellas con valores numéricamente mayores influyan desproporcionadamente en los modelos de Machine Learning. Al aplicar `MinMaxScaler`, todas las características se transforman al rango [0, 1], manteniendo su forma original pero haciendo que contribuyan de manera equilibrada al modelo. Esto mejora la estabilidad de los cálculos, acelera la convergencia y aumenta la precisión en algoritmos sensibles a la magnitud de los datos.

---
## 📊 FASE 4 (EDA): ANÁLISIS EXPLORATORIO
### PASO 7: Visualización de la Variable Objetivo y Distribuciones

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(datos_trabajo['pH'], kde=True, bins=30)
plt.title('Distribución del pH del vino tinto')
plt.xlabel('pH')
plt.ylabel('Frecuencia')
plt.tight_layout()
plt.show()

#### 📊 Análisis: Distribución del pH del vino tinto

- La distribución del pH muestra una forma aproximadamente normal, con la mayor concentración de valores entre 3.1 y 3.4.
- Esto indica que la mayoría de los vinos presentan una acidez moderada y estable.
- Esta ligera simetría implica que el pH es una variable adecuada para el análisis y probablemente no requiere transformaciones adicionales.

In [ ]:
plt.figure(figsize=(10, 6))
sns.countplot(
    data=datos_trabajo,
    y='quality',
    order=datos_trabajo['quality'].value_counts().sort_index().index
)
plt.title('Distribución de la calidad del vino')
plt.xlabel('Cantidad de vinos')
plt.ylabel('Calidad (quality)')
plt.tight_layout()
plt.show()

#### 📊 Análisis: Distribución de la calidad del vino

- La calidad del vino se concentra principalmente en los valores 5 y 6, mientras que las calidades extremas como 3, 4, 7 y 8 aparecen con baja frecuencia.
- Este claro desequilibrio de clases puede dificultar que un modelo predictivo aprenda patrones en los valores menos representados.
- Esto significa que el modelo tenderá a predecir valores centrales a menos que se utilicen técnicas de balanceo.

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(
    x='alcohol',
    y='pH',
    hue='quality',
    data=datos_trabajo,
    palette='viridis',
    alpha=0.7
)
plt.title('Relación entre Alcohol y pH coloreada por Calidad del Vino')
plt.xlabel('Alcohol (%)')
plt.ylabel('pH')
plt.tight_layout()
plt.show()

#### 📊 Análisis: Relación entre Alcohol y pH coloreada por Nivel de Calidad

- El gráfico muestra que no existe una relación lineal fuerte entre el alcohol y el pH, ya que los puntos aparecen dispersos sin una tendencia clara.
- Sin embargo, los vinos de mayor calidad tienden a concentrarse en valores ligeramente más altos de alcohol.
- Esto sugiere que, aunque el pH no está directamente relacionado con el alcohol, el contenido alcohólico podría influir más significativamente en la calidad del vino.

In [ ]:
valores = datos_trabajo['quality'].value_counts().sort_index()
labels = valores.index.astype(str)
explode = [0.005] * len(valores)

plt.figure(figsize=(8, 10))
plt.pie(
    valores,
    labels=labels,
    autopct='%1.1f%%',
    explode=explode,
    startangle=2190,
    pctdistance=1,
    labeldistance=1.1,
    textprops={'fontsize': 12}
)
plt.title('Proporción de Vinos por Nivel de Calidad', fontsize=14)
plt.tight_layout()
plt.show()

#### 📊 Análisis: Proporción de Vinos por Nivel de Calidad

- La gráfica de pastel confirma que las calidades 5 y 6 representan más del 80% de los vinos analizados.
- Las calidades altas (7 y 8) y bajas (3 y 4) tienen una presencia mínima, lo cual refuerza la existencia de un fuerte desequilibrio en los datos.
- Para el análisis predictivo, esto implica que un modelo puede presentar sesgos hacia los valores centrales si no se implementan técnicas de ajuste o balanceo.

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=datos_trabajo, x='alcohol')
plt.title('Boxplot del contenido de alcohol')
plt.xlabel('Alcohol (%)')
plt.tight_layout()
plt.show()

#### 📊 Análisis: Boxplot del Contenido de Alcohol

- El boxplot revela que la mayoría de los vinos tienen un contenido de alcohol entre aproximadamente 9.5% y 11.5%, con varios outliers que superan estos valores llegando hasta casi 15%.
- Estos valores extremos pueden influir significativamente en modelos sensibles, especialmente en algoritmos lineales.
- Esto resalta la importancia de aplicar un escalado adecuado y considerar técnicas para mitigar el impacto de los outliers.

### PASO 8: Correlación (Spearman)

In [ ]:
# Calcular matriz de correlación
correlacion = datos_trabajo.corr(method='spearman')

# Visualizar con heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(correlacion, annot=True, cmap='coolwarm',
            fmt='.2f', linewidths=0.5, center=0)
plt.title('Matriz de Correlación de Spearman', fontsize=16)
plt.tight_layout()
plt.show()

# Identificar correlaciones fuertes
print("\n=== CORRELACIONES MÁS FUERTES ===")

correlaciones_fuertes = []
columnas = correlacion.columns

for i in range(len(columnas)):
    for j in range(i + 1, len(columnas)):
        valor = correlacion.iloc[i, j]
        if abs(valor) >= 0.5:
            correlaciones_fuertes.append((columnas[i], columnas[j], valor))

correlaciones_ordenadas = sorted(
    correlaciones_fuertes, key=lambda x: abs(x[2]), reverse=True
)

for var1, var2, corr in correlaciones_ordenadas[:3]:
    print(f"{var1} ↔ {var2}: {corr:.3f}")

La matriz de correlación de Spearman permite identificar relaciones monótonas entre las variables del vino, siendo robusta ante valores atípicos y útil incluso cuando las relaciones no son lineales. En el dataset se observan correlaciones particularmente fuertes entre algunas variables, como la relación negativa entre *alcohol* y *density*, lo cual es esperable porque mayor alcohol disminuye la densidad del vino.

También destacan correlaciones positivas como *fixed acidity* con *citric acid*, indicando que ambas medidas químicas aumentan de forma similar. Asimismo, se observa que *alcohol* mantiene una correlación positiva con *quality*, sugiriendo que vinos con mayor graduación alcohólica tienden a recibir mejores puntuaciones.

### PASO 9: Test de Normalidad (Shapiro-Wilk)

In [ ]:
from scipy.stats import shapiro
import pandas as pd

def test_normalidad(dataframe, alpha=0.05):
    print("=== TEST DE SHAPIRO-WILK ===")
    print(f"Nivel de significancia (alpha): {alpha}\n")

    for col in dataframe.columns:
        if pd.api.types.is_numeric_dtype(dataframe[col]):
            data = dataframe[col].dropna()

            if len(data) >= 3:
                stat, p_value = shapiro(data)
                resultado = "NORMAL" if p_value > alpha else "NO NORMAL"

                print(f"\nVariable: {col}")
                print(f" Estadístico W: {stat:.4f}")
                print(f" P-valor: {p_value:.4f}")
                print(f" Conclusión: {resultado}")
                print("-" * 40)

# Aplicar el test a los datos normalizados
test_normalidad(datos_trabajo)

El test de Shapiro–Wilk aplicado a todas las variables numéricas del dataset indica que **ninguna de ellas sigue una distribución normal**, ya que en todos los casos el p-valor fue menor que el nivel de significancia (α = 0.05). Esto es completamente esperado, ya que las variables químicas suelen presentar distribuciones sesgadas y con colas largas, mientras que *quality* es una variable discreta y, por tanto, no puede ajustarse a una distribución normal continua.

Además, debido al gran tamaño de la muestra (1,599 observaciones), el test de Shapiro–Wilk se vuelve muy sensible y detecta incluso pequeñas desviaciones respecto a la normalidad. Estas conclusiones refuerzan la decisión de utilizar técnicas estadísticas robustas y métodos de correlación como Spearman, que no dependen del supuesto de normalidad.

---
## 🤖 FASE 4: REGRESIÓN LINEAL SIMPLE
### Celda 9 — Split Train/Test (predictor: `citric acid`)

In [ ]:
from sklearn.model_selection import train_test_split

# Variable predictora y objetivo para regresión simple
X_simple = datos_trabajo[['citric acid']]
y_simple  = datos_trabajo['quality']

# División 80 % entrenamiento / 20 % prueba
X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_simple, y_simple, test_size=0.2, random_state=42
)

print("=== SPLIT TRAIN / TEST (Regresión Simple) ===")
print(f"Total de muestras  : {len(X_simple)}")
print(f"Entrenamiento (80%): {len(X_train_s)} muestras")
print(f"Prueba        (20%): {len(X_test_s)} muestras")
print(f"\nPredictor : citric acid")
print(f"Objetivo  : quality")

# Vista rápida de los conjuntos
print("\n--- Estadísticos X_train_s ---")
display(X_train_s.describe())
print("--- Estadísticos X_test_s ---")
display(X_test_s.describe())

#### 📌 Análisis — Celda 9: Split Train/Test

Se seleccionó **`citric acid`** como único predictor para el modelo de regresión lineal **simple**. Esta elección responde a que el ácido cítrico es uno de los componentes que incide directamente en la frescura y equilibrio sensorial del vino, y su relación con la calidad puede evaluarse de forma aislada antes de incorporar más variables.

La partición 80/20 es una práctica estándar en Machine Learning:
- **80 % de entrenamiento** garantiza que el modelo disponga de suficientes ejemplos para aprender la tendencia.
- **20 % de prueba** permite evaluar la capacidad de generalización del modelo en datos no vistos.

La semilla `random_state=42` asegura que los resultados sean **reproducibles** en cualquier ejecución posterior.

### Celda 10 — Entrenamiento del Modelo Simple

In [ ]:
from sklearn.linear_model import LinearRegression

# Crear y entrenar el modelo de regresión simple
modelo_simple = LinearRegression()
modelo_simple.fit(X_train_s, y_train_s)

# Predicciones sobre el conjunto de prueba
y_pred_s = modelo_simple.predict(X_test_s)

print("=== PARÁMETROS DEL MODELO SIMPLE ===")
print(f"Intercepto (β₀)        : {modelo_simple.intercept_:.4f}")
print(f"Coeficiente β₁ (citric acid): {modelo_simple.coef_[0]:.4f}")
print(f"\nEcuación aprendida:")
print(f"  quality = {modelo_simple.intercept_:.4f} + ({modelo_simple.coef_[0]:.4f}) × citric_acid")

#### 📌 Análisis — Celda 10: Entrenamiento del Modelo Simple

El modelo de **regresión lineal simple** busca la mejor recta que relaciona `citric acid` con `quality` minimizando el error cuadrático. Los parámetros clave son:

- **β₀ (intercepto):** valor esperado de `quality` cuando `citric acid = 0`. En el espacio normalizado, este valor representa la calidad base estimada.
- **β₁ (pendiente):** variación esperada en `quality` por cada unidad de incremento en `citric acid`. Un coeficiente positivo indicaría que mayor acidez cítrica se asocia a mayor calidad percibida, y uno negativo lo contrario.

Es importante recordar que estamos trabajando con **datos normalizados** (rango [0, 1]), por lo que los coeficientes no están en las unidades originales del vino.

### Celda 11 — Gráfica de la Recta de Regresión (Visualización del Ajuste)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Rango de valores para dibujar la recta
x_range = np.linspace(X_simple['citric acid'].min(), X_simple['citric acid'].max(), 200).reshape(-1, 1)
y_range = modelo_simple.predict(x_range)

plt.figure(figsize=(10, 6))
plt.scatter(X_test_s, y_test_s, alpha=0.4, color='steelblue', label='Datos de prueba', s=25)
plt.plot(x_range, y_range, color='crimson', linewidth=2.5, label='Recta de regresión')
plt.xlabel('Citric Acid (normalizado)', fontsize=12)
plt.ylabel('Quality (normalizado)', fontsize=12)
plt.title('Regresión Lineal Simple: Citric Acid → Quality', fontsize=14)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

#### 📌 Análisis — Celda 11: Recta de Regresión

La visualización superpone los **puntos de prueba** con la **recta de regresión ajustada**. Elementos clave a observar:

- **Dispersión de puntos:** la alta dispersión vertical alrededor de la recta indica que `citric acid` por sí solo explica **una fracción pequeña** de la variabilidad de `quality`. Esto es esperable en un fenómeno tan complejo como la calidad sensorial del vino.
- **Pendiente de la recta:** permite ver la dirección de la relación entre las dos variables. Una pendiente suave o casi plana confirmaría que el ácido cítrico no es un predictor fuerte en solitario.
- **Valores discretos de quality:** al estar normalizada, la variable objetivo toma solo un conjunto finito de valores, lo que genera las "bandas horizontales" visibles en el gráfico. Esto es una característica intrínseca del dataset.

### Celda 12 — Métricas del Modelo Simple (R², MSE, RMSE, MAE)

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import numpy as np

mse_s   = mean_squared_error(y_test_s, y_pred_s)
rmse_s  = np.sqrt(mse_s)
mae_s   = mean_absolute_error(y_test_s, y_pred_s)
r2_s    = r2_score(y_test_s, y_pred_s)

print("=" * 45)
print("  MÉTRICAS — MODELO DE REGRESIÓN SIMPLE")
print("=" * 45)
print(f"  R²   (Coef. de determinación) : {r2_s:.4f}")
print(f"  MSE  (Error cuadrático medio)  : {mse_s:.4f}")
print(f"  RMSE (Raíz del MSE)            : {rmse_s:.4f}")
print(f"  MAE  (Error absoluto medio)    : {mae_s:.4f}")
print("=" * 45)
print(f"\nInterpretación R²: el modelo explica el {r2_s*100:.2f}% de la")
print("variabilidad de 'quality' usando solo 'citric acid'.")

#### 📌 Análisis — Celda 12: Métricas del Modelo Simple

Las cuatro métricas ofrecen una visión complementaria del desempeño del modelo:

| Métrica | Descripción | Interpretación ideal |
|---------|-------------|----------------------|
| **R²** | Proporción de varianza explicada | Cuanto más cercano a 1, mejor |
| **MSE** | Penaliza errores grandes al elevarlos al cuadrado | Cuanto menor, mejor |
| **RMSE** | Mismas unidades que la variable objetivo | Cuanto menor, mejor |
| **MAE** | Error promedio absoluto, más robusto a outliers | Cuanto menor, mejor |

Un **R² bajo** para la regresión simple confirma que un solo predictor no es suficiente para capturar la complejidad de la calidad del vino. Esto motiva el paso a la **regresión múltiple**, donde se incorporarán todas las variables fisicoquímicas disponibles.

### Celda 13 — Resumen OLS con `statsmodels` (Análisis Estadístico Detallado)

In [ ]:
import statsmodels.api as sm

# statsmodels requiere añadir la constante (intercepto) manualmente
X_train_sm = sm.add_constant(X_train_s)

# Ajustar modelo OLS sobre el conjunto de entrenamiento
ols_simple = sm.OLS(y_train_s, X_train_sm).fit()

# Mostrar el resumen completo
print(ols_simple.summary())

#### 📌 Análisis — Celda 13: Resumen OLS (Modelo Simple)

El resumen de `statsmodels` proporciona información estadística que `sklearn` no entrega directamente:

- **R² y R² ajustado:** el R² ajustado penaliza la adición de predictores innecesarios; en la regresión simple ambos coinciden aproximadamente.
- **Coeficientes y p-valores:** el p-valor del coeficiente de `citric acid` indica si la relación con `quality` es **estadísticamente significativa** (p < 0.05). Si es mayor a 0.05, el predictor no aporta información significativa al modelo.
- **F-statistic:** prueba si el modelo en conjunto explica significativamente la varianza de la variable objetivo.
- **AIC / BIC:** criterios de información útiles para comparar modelos; valores menores indican mejor ajuste relativo.
- **Intervalos de confianza (95%):** rango probable del verdadero valor del coeficiente en la población.

Este análisis permite tomar decisiones fundamentadas sobre la inclusión o exclusión de variables y la significancia del modelo.

### Celda 14 — Normalidad de Residuos (Histograma + Q-Q Plot + Shapiro-Wilk)

In [ ]:
import scipy.stats as stats
import matplotlib.pyplot as plt

# Calcular residuos del conjunto de prueba
residuos_s = y_test_s - y_pred_s

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Histograma de residuos ──────────────────────────────────────────────────
axes[0].hist(residuos_s, bins=30, edgecolor='black', color='steelblue', alpha=0.75)
axes[0].axvline(0, color='red', linestyle='--', linewidth=1.5, label='Media = 0')
axes[0].set_title('Histograma de Residuos — Modelo Simple', fontsize=13)
axes[0].set_xlabel('Residuos')
axes[0].set_ylabel('Frecuencia')
axes[0].legend()

# ── Q-Q Plot ────────────────────────────────────────────────────────────────
stats.probplot(residuos_s, dist="norm", plot=axes[1])
axes[1].set_title('Q-Q Plot de Residuos — Modelo Simple', fontsize=13)

plt.tight_layout()
plt.show()

# ── Test de Shapiro-Wilk sobre los residuos ─────────────────────────────────
stat_res, p_res = stats.shapiro(residuos_s)
print("\n=== TEST DE SHAPIRO-WILK SOBRE RESIDUOS ===")
print(f"  Estadístico W : {stat_res:.4f}")
print(f"  P-valor        : {p_res:.4f}")
conclusion = "NORMAL (p > 0.05)" if p_res > 0.05 else "NO NORMAL (p ≤ 0.05)"
print(f"  Conclusión     : Los residuos son {conclusion}")

#### 📌 Análisis — Celda 14: Normalidad de Residuos

Uno de los supuestos clave de la regresión lineal clásica es que los **residuos sigan una distribución normal con media cero**. Se utilizan tres herramientas complementarias para verificarlo:

1. **Histograma de residuos:** permite observar la forma de la distribución. Una curva simétrica y centrada en 0 sugiere normalidad. Sesgos o bimodalidades indican violaciones al supuesto.

2. **Q-Q Plot:** compara los cuantiles de los residuos con los cuantiles teóricos de una distribución normal. Si los puntos siguen la diagonal roja, los residuos son normales. Desviaciones en los extremos indican colas pesadas (leptocurtosis) o distribución asimétrica.

3. **Test de Shapiro-Wilk:** contraste formal de normalidad. Si el p-valor es mayor que 0.05, no se rechaza la hipótesis de normalidad. En muestras grandes (como este dataset), el test es muy sensible y puede detectar desviaciones mínimas irrelevantes en la práctica.

**Interpretación práctica:** si los residuos no son normales, las inferencias estadísticas (intervalos de confianza, p-valores) deben interpretarse con precaución, aunque el modelo sigue siendo válido para predicción.

### Celda 15 — Homocedasticidad (Residuos vs Ajustados + Breusch-Pagan)

In [ ]:
from statsmodels.stats.diagnostic import het_breuschpagan
import statsmodels.api as sm
import matplotlib.pyplot as plt
import numpy as np

# ── Gráfico: Residuos vs Valores Ajustados ──────────────────────────────────
plt.figure(figsize=(10, 5))
plt.scatter(y_pred_s, residuos_s, alpha=0.4, color='steelblue', s=20)
plt.axhline(0, color='red', linestyle='--', linewidth=1.5)
plt.xlabel('Valores Ajustados (ŷ)', fontsize=12)
plt.ylabel('Residuos', fontsize=12)
plt.title('Homocedasticidad: Residuos vs Valores Ajustados (Modelo Simple)', fontsize=13)
plt.tight_layout()
plt.show()

# ── Test de Breusch-Pagan ────────────────────────────────────────────────────
X_test_sm_bp = sm.add_constant(X_test_s)
bp_test = het_breuschpagan(residuos_s, X_test_sm_bp)
labels_bp = ['LM Statistic', 'LM p-value', 'F-statistic', 'F p-value']

print("\n=== TEST DE BREUSCH-PAGAN (Homocedasticidad) ===")
for label, value in zip(labels_bp, bp_test):
    print(f"  {label:20s}: {value:.4f}")

p_bp = bp_test[1]
if p_bp < 0.05:
    print("\n  → Conclusión: Se detecta HETEROCEDASTICIDAD (p ≤ 0.05).")
    print("    La varianza de los residuos NO es constante.")
else:
    print("\n  → Conclusión: No se rechaza HOMOCEDASTICIDAD (p > 0.05).")
    print("    La varianza de los residuos es aproximadamente constante.")

#### 📌 Análisis — Celda 15: Homocedasticidad

La **homocedasticidad** (varianza constante de los residuos) es otro supuesto fundamental de la regresión lineal. Se evalúa con dos enfoques:

**1. Gráfico de Residuos vs Valores Ajustados:**
- Distribución **aleatoria** de puntos alrededor de la línea horizontal en 0 → homocedasticidad.
- Patrón en "abanico" (dispersión que crece o decrece) → heterocedasticidad.
- Patrones curvos → posible no linealidad en el modelo.

**2. Test de Breusch-Pagan:**
- **H₀:** los residuos son homocedásticos (varianza constante).
- **H₁:** existe heterocedasticidad.
- Si p-valor < 0.05 → se rechaza H₀ y se confirma heterocedasticidad.

**Consecuencias prácticas:** la heterocedasticidad no invalida las estimaciones de coeficientes, pero sí afecta la eficiencia de los estimadores y la validez de los tests de significancia. En ese caso, se podría recurrir a errores estándar robustos (tipo Huber-White) o transformaciones de la variable objetivo.

---
## 🤖 FASE 5: REGRESIÓN MÚLTIPLE + DIAGNÓSTICO
### Celda 16 — Entrenamiento del Modelo Múltiple OLS

In [ ]:
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

# ── Variables predictoras y objetivo ────────────────────────────────────────
X_mult = datos_trabajo.drop('quality', axis=1)
y_mult = datos_trabajo['quality']

# División 80/20
X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_mult, y_mult, test_size=0.2, random_state=42
)

# ── sklearn: para predicciones ──────────────────────────────────────────────
modelo_multiple = LinearRegression()
modelo_multiple.fit(X_train_m, y_train_m)
y_pred_m = modelo_multiple.predict(X_test_m)

# ── statsmodels OLS: para análisis estadístico ──────────────────────────────
X_train_sm_m = sm.add_constant(X_train_m)
ols_multiple = sm.OLS(y_train_m, X_train_sm_m).fit()

print("=== MODELO MÚLTIPLE ENTRENADO ===")
print(f"Número de predictores  : {X_mult.shape[1]}")
print(f"Muestras entrenamiento : {len(X_train_m)}")
print(f"Muestras prueba        : {len(X_test_m)}")
print(f"\nVariables incluidas:")
for col in X_mult.columns:
    print(f"  • {col}")

#### 📌 Análisis — Celda 16: Entrenamiento del Modelo Múltiple

El **modelo de regresión múltiple** incorpora **todas las variables fisicoquímicas** disponibles como predictores de `quality`. A diferencia del modelo simple, este modelo puede capturar la contribución conjunta de múltiples factores que influyen en la calidad del vino.

Se utilizan dos implementaciones en paralelo:
- **`sklearn.LinearRegression`:** optimizado para predicción y cálculo de métricas.
- **`statsmodels.OLS`:** proporciona el detalle estadístico completo (p-valores, intervalos de confianza, tests diagnósticos).

La misma partición 80/20 con `random_state=42` garantiza una comparación justa y reproducible con el modelo simple.

### Celda 17 — Gráfica Comparativa de Coeficientes (Importancia de Variables)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# DataFrame de coeficientes ordenados por valor absoluto
coef_df = pd.DataFrame({
    'Variable'   : X_mult.columns,
    'Coeficiente': modelo_multiple.coef_
}).sort_values('Coeficiente', key=abs, ascending=True)

# Asignar colores según signo del coeficiente
colores = ['#d62728' if c < 0 else '#2ca02c' for c in coef_df['Coeficiente']]

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(coef_df['Variable'], coef_df['Coeficiente'], color=colores, edgecolor='black', alpha=0.85)
ax.axvline(0, color='black', linewidth=1.2, linestyle='--')
ax.set_xlabel('Valor del Coeficiente (datos normalizados)', fontsize=12)
ax.set_title('Importancia de Variables — Regresión Múltiple\n(verde = positivo | rojo = negativo)', fontsize=13)

# Anotar valores en las barras
for bar, val in zip(bars, coef_df['Coeficiente']):
    ax.text(
        val + (0.002 if val >= 0 else -0.002),
        bar.get_y() + bar.get_height() / 2,
        f'{val:.3f}', va='center',
        ha='left' if val >= 0 else 'right', fontsize=9
    )

plt.tight_layout()
plt.show()

print("\n=== COEFICIENTES ORDENADOS POR MAGNITUD ===")
display(coef_df.sort_values('Coeficiente', key=abs, ascending=False).reset_index(drop=True))

#### 📌 Análisis — Celda 17: Importancia de Variables

El gráfico de coeficientes ofrece una lectura visual inmediata de **qué variables tienen mayor impacto** sobre la predicción de calidad:

- **Barras verdes (coeficiente positivo):** un aumento en esa variable se asocia con mayor calidad predicha. Variables como `alcohol` y `sulphates` suelen destacar aquí.
- **Barras rojas (coeficiente negativo):** un aumento en esa variable se asocia con menor calidad predicha. `volatile acidity` y `chlorides` tienden a presentar esta relación.
- **Magnitud del coeficiente:** cuanto mayor sea la barra (en valor absoluto), mayor es la influencia proporcional de la variable sobre la predicción.

> **Importante:** los coeficientes son comparables entre sí porque los datos fueron normalizados en el rango [0, 1]. Sin normalización, la magnitud dependería de las unidades de cada variable.

### Celda 18 — Métricas del Modelo Múltiple

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import numpy as np

mse_m  = mean_squared_error(y_test_m, y_pred_m)
rmse_m = np.sqrt(mse_m)
mae_m  = mean_absolute_error(y_test_m, y_pred_m)
r2_m   = r2_score(y_test_m, y_pred_m)

print("=" * 47)
print("  MÉTRICAS — MODELO DE REGRESIÓN MÚLTIPLE")
print("=" * 47)
print(f"  R²   (Coef. de determinación) : {r2_m:.4f}")
print(f"  MSE  (Error cuadrático medio)  : {mse_m:.4f}")
print(f"  RMSE (Raíz del MSE)            : {rmse_m:.4f}")
print(f"  MAE  (Error absoluto medio)    : {mae_m:.4f}")
print("=" * 47)
print(f"\nEl modelo explica el {r2_m*100:.2f}% de la variabilidad")
print("de 'quality' usando TODAS las variables fisicoquímicas.")

#### 📌 Análisis — Celda 18: Métricas del Modelo Múltiple

Al incorporar todas las variables del dataset, se espera una mejora significativa respecto al modelo simple:

- **R²:** el salto en R² respecto al modelo simple cuantifica el beneficio de añadir más predictores. Sin embargo, un R² más alto no siempre significa un modelo mejor si hay sobreajuste.
- **RMSE y MAE:** su reducción confirma que el modelo múltiple comete errores menores en promedio. Al estar en escala normalizada, el RMSE representa el error como fracción del rango total de calidad.
- **Limitación esperada:** aunque el modelo múltiple mejora al simple, un R² moderado (alrededor de 0.35–0.40) indica que la relación entre las variables fisicoquímicas y la calidad percibida **no es perfectamente lineal**, y posiblemente requiera modelos más complejos para ser capturada con mayor precisión.

### Celda 19 — Resumen OLS `statsmodels` (Comparativa Técnica frente al Simple)

In [ ]:
# Mostrar resumen completo del modelo múltiple
print(ols_multiple.summary())

# Comparativa rápida de métricas estadísticas: Simple vs Múltiple
print("\n" + "=" * 55)
print("  COMPARATIVA ESTADÍSTICA: SIMPLE vs MÚLTIPLE (OLS)")
print("=" * 55)
print(f"  {'Métrica':<30} {'Simple':>10} {'Múltiple':>10}")
print("-" * 55)

# OLS sobre conjunto de entrenamiento del modelo simple
print(f"  {'R² (train)':<30} {ols_simple.rsquared:>10.4f} {ols_multiple.rsquared:>10.4f}")
print(f"  {'R² ajustado (train)':<30} {ols_simple.rsquared_adj:>10.4f} {ols_multiple.rsquared_adj:>10.4f}")
print(f"  {'AIC':<30} {ols_simple.aic:>10.2f} {ols_multiple.aic:>10.2f}")
print(f"  {'BIC':<30} {ols_simple.bic:>10.2f} {ols_multiple.bic:>10.2f}")
print(f"  {'F-statistic (p-value)':<30} {ols_simple.f_pvalue:>10.4f} {ols_multiple.f_pvalue:>10.4f}")
print("=" * 55)

#### 📌 Análisis — Celda 19: Resumen OLS y Comparativa Simple vs Múltiple

El resumen OLS del modelo múltiple permite evaluar la **significancia individual de cada predictor**:

- **p-valores individuales:** variables con p < 0.05 son estadísticamente significativas. Aquellas con p > 0.05 podrían eliminarse sin pérdida relevante de poder predictivo.
- **R² ajustado:** a diferencia del R² simple, penaliza la adición de predictores irrelevantes. Es la métrica correcta para comparar modelos con distinto número de variables.
- **AIC y BIC:** criterios de información. El modelo con **menor AIC/BIC** es preferido. Se espera que el modelo múltiple tenga un AIC/BIC más bajo que el simple, confirmando que el costo de añadir variables está justificado por la ganancia en ajuste.
- **F-statistic global:** prueba si el conjunto de predictores en su totalidad explica significativamente la varianza de la variable objetivo.

### Celda 20 — VIF: Diagnóstico de Multicolinealidad

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

# Calcular VIF para cada predictor
X_vif = sm.add_constant(X_train_m)

vif_data = pd.DataFrame()
vif_data['Variable'] = X_vif.columns
vif_data['VIF'] = [
    variance_inflation_factor(X_vif.values, i)
    for i in range(X_vif.shape[1])
]

# Excluir constante de la visualización
vif_data = vif_data[vif_data['Variable'] != 'const'].sort_values('VIF', ascending=False)

# Clasificar por nivel de riesgo
def clasificar_vif(v):
    if v < 5:   return 'Bajo ✅'
    elif v < 10: return 'Moderado ⚠️'
    else:        return 'Alto 🔴'

vif_data['Nivel'] = vif_data['VIF'].apply(clasificar_vif)

print("=== FACTOR DE INFLACIÓN DE LA VARIANZA (VIF) ===")
display(vif_data.reset_index(drop=True))

# Gráfico de barras VIF
colores_vif = ['#d62728' if v >= 10 else '#ff7f0e' if v >= 5 else '#2ca02c'
               for v in vif_data['VIF']]

plt.figure(figsize=(10, 6))
plt.barh(vif_data['Variable'], vif_data['VIF'], color=colores_vif, edgecolor='black', alpha=0.85)
plt.axvline(5,  color='orange', linestyle='--', linewidth=1.5, label='Umbral moderado (VIF=5)')
plt.axvline(10, color='red',    linestyle='--', linewidth=1.5, label='Umbral alto (VIF=10)')
plt.xlabel('VIF', fontsize=12)
plt.title('Diagnóstico de Multicolinealidad — VIF por Variable', fontsize=13)
plt.legend(fontsize=10)
plt.tight_layout()
plt.show()

#### 📌 Análisis — Celda 20: VIF y Multicolinealidad

El **Factor de Inflación de la Varianza (VIF)** mide cuánto se infla la varianza de un coeficiente debido a su correlación con otros predictores. Es la herramienta estándar para diagnosticar **multicolinealidad** en regresión múltiple.

| Valor VIF | Interpretación |
|-----------|----------------|
| 1 | Sin correlación con otras variables |
| 1 – 5 | Multicolinealidad baja (aceptable) |
| 5 – 10 | Multicolinealidad moderada (revisar) |
| > 10 | Multicolinealidad alta (problemática) |

**Efectos de la multicolinealidad alta:**
- Los coeficientes se vuelven inestables y difíciles de interpretar.
- Los errores estándar se inflan, reduciendo la significancia estadística.
- El modelo pierde capacidad explicativa individual por variable.

**Acciones correctivas:** eliminar una de las variables correlacionadas, aplicar PCA (Análisis de Componentes Principales) o usar regularización (Ridge, Lasso).

### Celda 21 — Diagnósticos Completos del Modelo Múltiple (Normalidad y Homocedasticidad)

In [ ]:
import scipy.stats as stats
from statsmodels.stats.diagnostic import het_breuschpagan
import statsmodels.api as sm
import matplotlib.pyplot as plt

# Residuos del modelo múltiple
residuos_m = y_test_m.values - y_pred_m

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Diagnósticos del Modelo de Regresión Múltiple', fontsize=15, fontweight='bold')

# ── 1. Histograma de residuos ───────────────────────────────────────────────
axes[0, 0].hist(residuos_m, bins=30, edgecolor='black', color='steelblue', alpha=0.75)
axes[0, 0].axvline(0, color='red', linestyle='--', linewidth=1.5)
axes[0, 0].set_title('Histograma de Residuos')
axes[0, 0].set_xlabel('Residuos')
axes[0, 0].set_ylabel('Frecuencia')

# ── 2. Q-Q Plot ─────────────────────────────────────────────────────────────
stats.probplot(residuos_m, dist="norm", plot=axes[0, 1])
axes[0, 1].set_title('Q-Q Plot de Residuos')

# ── 3. Residuos vs Valores Ajustados ────────────────────────────────────────
axes[1, 0].scatter(y_pred_m, residuos_m, alpha=0.4, color='darkorange', s=20)
axes[1, 0].axhline(0, color='red', linestyle='--', linewidth=1.5)
axes[1, 0].set_xlabel('Valores Ajustados (ŷ)')
axes[1, 0].set_ylabel('Residuos')
axes[1, 0].set_title('Residuos vs Valores Ajustados')

# ── 4. Scale-Location (√|Residuos estandarizados| vs Ajustados) ─────────────
residuos_std = np.sqrt(np.abs(residuos_m / residuos_m.std()))
axes[1, 1].scatter(y_pred_m, residuos_std, alpha=0.4, color='purple', s=20)
axes[1, 1].set_xlabel('Valores Ajustados (ŷ)')
axes[1, 1].set_ylabel('√|Residuos Estandarizados|')
axes[1, 1].set_title('Scale-Location (Homocedasticidad)')

plt.tight_layout()
plt.show()

# ── Tests formales ───────────────────────────────────────────────────────────
print("\n=== TEST SHAPIRO-WILK (Normalidad de Residuos) ===")
stat_m, p_m = stats.shapiro(residuos_m)
print(f"  W = {stat_m:.4f} | p-valor = {p_m:.4f}")
print(f"  → {'NORMAL' if p_m > 0.05 else 'NO NORMAL'} (α = 0.05)")

print("\n=== TEST BREUSCH-PAGAN (Homocedasticidad) ===")
X_test_sm_m = sm.add_constant(X_test_m)
bp_m = het_breuschpagan(residuos_m, X_test_sm_m)
labels_bp = ['LM Statistic', 'LM p-value', 'F-statistic', 'F p-value']
for label, value in zip(labels_bp, bp_m):
    print(f"  {label:20s}: {value:.4f}")
p_bp_m = bp_m[1]
print(f"\n  → {'HETEROCEDASTICIDAD detectada' if p_bp_m < 0.05 else 'No se rechaza HOMOCEDASTICIDAD'} (p = {p_bp_m:.4f})")

#### 📌 Análisis — Celda 21: Diagnósticos Completos del Modelo Múltiple

Se aplica un **panel diagnóstico de 4 gráficas** que cubre los principales supuestos de la regresión lineal clásica:

1. **Histograma de residuos:** evalúa visualmente si los residuos siguen una distribución campaniforme centrada en cero.
2. **Q-Q Plot:** contraste gráfico de normalidad. Los puntos deben alinearse sobre la diagonal.
3. **Residuos vs Ajustados:** detecta heterocedasticidad y patrones no lineales. Se busca dispersión aleatoria alrededor del cero.
4. **Scale-Location:** variante que usa la raíz cuadrada de los residuos estandarizados para detectar con mayor claridad si la varianza crece o decrece.

Los tests formales complementan el análisis visual:
- **Shapiro-Wilk sobre residuos:** contraste de normalidad.
- **Breusch-Pagan:** contraste de homocedasticidad.

En general, la regresión lineal es **robusta** ante violaciones moderadas de estos supuestos, especialmente con muestras grandes como la de este dataset.

---
## 📊 FASE 6: DIAGNÓSTICOS CONSOLIDADOS Y COMPARACIÓN DE MODELOS
### Celda 22 — Tabla Comparativa de Métricas (Simple vs. Múltiple)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# Calcular métricas para ambos modelos
metricas = {
    'Modelo': ['Regresión Simple\n(citric acid)', 'Regresión Múltiple\n(todas las variables)'],
    'R²':   [r2_score(y_test_s, y_pred_s),          r2_score(y_test_m, y_pred_m)],
    'MSE':  [mean_squared_error(y_test_s, y_pred_s), mean_squared_error(y_test_m, y_pred_m)],
    'RMSE': [np.sqrt(mean_squared_error(y_test_s, y_pred_s)), np.sqrt(mean_squared_error(y_test_m, y_pred_m))],
    'MAE':  [mean_absolute_error(y_test_s, y_pred_s), mean_absolute_error(y_test_m, y_pred_m)],
    'N predictores': [1, X_mult.shape[1]],
    'R² ajustado (train)': [ols_simple.rsquared_adj, ols_multiple.rsquared_adj],
    'AIC': [ols_simple.aic, ols_multiple.aic],
    'BIC': [ols_simple.bic, ols_multiple.bic],
}

df_metricas = pd.DataFrame(metricas).set_index('Modelo')
print("=" * 65)
print("    TABLA COMPARATIVA DE MÉTRICAS — SIMPLE vs MÚLTIPLE")
print("=" * 65)
display(df_metricas.T)

# ── Visualización: Barras comparativas para R², RMSE y MAE ──────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Comparación de Métricas: Regresión Simple vs Múltiple', fontsize=14, fontweight='bold')

modelos = ['Simple', 'Múltiple']
metricas_plot = [
    ('R²',   [r2_score(y_test_s, y_pred_s), r2_score(y_test_m, y_pred_m)],   ['#5fa8d3', '#e76f51']),
    ('RMSE', [np.sqrt(mean_squared_error(y_test_s, y_pred_s)), np.sqrt(mean_squared_error(y_test_m, y_pred_m))], ['#5fa8d3', '#e76f51']),
    ('MAE',  [mean_absolute_error(y_test_s, y_pred_s), mean_absolute_error(y_test_m, y_pred_m)], ['#5fa8d3', '#e76f51']),
]

for ax, (nombre, valores, colores) in zip(axes, metricas_plot):
    bars = ax.bar(modelos, valores, color=colores, edgecolor='black', alpha=0.85, width=0.5)
    ax.set_title(nombre, fontsize=13)
    ax.set_ylabel(nombre)
    for bar, val in zip(bars, valores):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                f'{val:.4f}', ha='center', va='bottom', fontsize=11)

plt.tight_layout()
plt.show()

#### 📌 Análisis — Celda 22: Tabla Comparativa de Métricas

La comparación directa entre el modelo simple y el múltiple permite cuantificar el **beneficio real de incorporar más predictores**:

| Aspecto | Esperado en Regresión Simple | Esperado en Regresión Múltiple |
|---------|------------------------------|--------------------------------|
| R² | Bajo (~0.02–0.10) | Moderado (~0.35–0.45) |
| RMSE | Mayor (más error) | Menor (menos error) |
| MAE | Mayor | Menor |
| AIC/BIC | Mayor | Menor |
| Complejidad | Mínima | Moderada |

El gráfico de barras facilita la comparación visual de las tres métricas principales:
- **R²:** mayor es mejor. La mejora del modelo múltiple es evidente.
- **RMSE y MAE:** menor es mejor. La reducción del error confirma la superioridad del modelo múltiple para la predicción.

Esta comparativa justifica objetivamente el uso del modelo múltiple como preferido para la predicción de calidad del vino.

### Celda 23 — Gráfica de Predicciones vs Valores Reales (Validación Final del Desempeño)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Validación Final: Predicciones vs Valores Reales', fontsize=15, fontweight='bold')

# ── Modelo Simple ─────────────────────────────────────────────────────────
min_s = min(y_test_s.min(), y_pred_s.min())
max_s = max(y_test_s.max(), y_pred_s.max())

axes[0].scatter(y_test_s, y_pred_s, alpha=0.5, color='steelblue', s=25, label='Predicciones')
axes[0].plot([min_s, max_s], [min_s, max_s], 'r--', linewidth=2, label='Predicción perfecta')
axes[0].set_xlabel('Valores Reales', fontsize=12)
axes[0].set_ylabel('Valores Predichos', fontsize=12)
axes[0].set_title(f'Modelo Simple\nR² = {r2_score(y_test_s, y_pred_s):.4f}', fontsize=13)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# ── Modelo Múltiple ──────────────────────────────────────────────────────
min_m = min(y_test_m.min(), y_pred_m.min())
max_m = max(y_test_m.max(), y_pred_m.max())

axes[1].scatter(y_test_m, y_pred_m, alpha=0.5, color='darkorange', s=25, label='Predicciones')
axes[1].plot([min_m, max_m], [min_m, max_m], 'r--', linewidth=2, label='Predicción perfecta')
axes[1].set_xlabel('Valores Reales', fontsize=12)
axes[1].set_ylabel('Valores Predichos', fontsize=12)
axes[1].set_title(f'Modelo Múltiple\nR² = {r2_score(y_test_m, y_pred_m):.4f}', fontsize=13)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# ── Distribución de errores de predicción ─────────────────────────────────
fig2, ax2 = plt.subplots(figsize=(12, 5))
ax2.hist(y_test_s.values - y_pred_s, bins=30, alpha=0.6, color='steelblue',
         edgecolor='black', label=f'Simple  | MAE={mean_absolute_error(y_test_s, y_pred_s):.4f}')
ax2.hist(y_test_m.values - y_pred_m, bins=30, alpha=0.6, color='darkorange',
         edgecolor='black', label=f'Múltiple | MAE={mean_absolute_error(y_test_m, y_pred_m):.4f}')
ax2.axvline(0, color='red', linestyle='--', linewidth=1.5, label='Error = 0')
ax2.set_xlabel('Error de Predicción (Real − Predicho)', fontsize=12)
ax2.set_ylabel('Frecuencia', fontsize=12)
ax2.set_title('Distribución de Errores de Predicción: Simple vs Múltiple', fontsize=13)
ax2.legend(fontsize=11)
plt.tight_layout()
plt.show()

print("\n=== RESUMEN FINAL DE COMPARACIÓN ===")
print(f"  Modelo Simple   → R²: {r2_score(y_test_s, y_pred_s):.4f} | RMSE: {np.sqrt(mean_squared_error(y_test_s, y_pred_s)):.4f} | MAE: {mean_absolute_error(y_test_s, y_pred_s):.4f}")
print(f"  Modelo Múltiple → R²: {r2_score(y_test_m, y_pred_m):.4f} | RMSE: {np.sqrt(mean_squared_error(y_test_m, y_pred_m)):.4f} | MAE: {mean_absolute_error(y_test_m, y_pred_m):.4f}")
print(f"\n  Mejora en R²  : +{(r2_score(y_test_m, y_pred_m) - r2_score(y_test_s, y_pred_s))*100:.2f} puntos porcentuales")
print(f"  Reducción RMSE: -{(np.sqrt(mean_squared_error(y_test_s, y_pred_s)) - np.sqrt(mean_squared_error(y_test_m, y_pred_m)))*100:.2f}%")

#### 📌 Análisis — Celda 23: Validación Final del Desempeño

Esta celda consolida la validación visual y cuantitativa del desempeño de ambos modelos mediante tres visualizaciones:

**1. Gráficos de Dispersión: Predicciones vs Valores Reales**
- La **línea roja discontinua** representa la predicción perfecta (real = predicho).
- Cuanto más cerca estén los puntos de esa línea, mejor es el modelo.
- El modelo simple mostrará puntos muy dispersos; el múltiple se acercará más a la diagonal.

**2. Distribución de Errores (Histograma superpuesto)**
- Permite comparar en un solo gráfico cómo se distribuyen los errores de ambos modelos.
- Una distribución más estrecha y centrada en 0 indica menor error y mejor precisión.
- El modelo múltiple debería mostrar una distribución más concentrada alrededor del cero.

**3. Resumen numérico final**
- Cuantifica exactamente la mejora obtenida al pasar del modelo simple al múltiple en términos de R² y RMSE.

**Conclusión general:** aunque el modelo múltiple es claramente superior al simple, el R² moderado obtenido sugiere que la calidad del vino involucra factores no capturados linealmente. Se recomienda explorar modelos de ensamble (Random Forest, Gradient Boosting) o enfoques de clasificación para mejorar la predicción.

---
## ✅ CONCLUSIONES FINALES

### 1. Resumen del análisis
El dataset analizado contiene información química de 1,599 vinos tintos y su calificación de calidad. A lo largo de las 6 fases del análisis se realizaron: carga y revisión del dataset, introducción de valores faltantes e imputación, normalización, visualización de distribuciones, análisis de correlación con Spearman, pruebas de normalidad con Shapiro–Wilk, regresión lineal simple con `citric acid` como predictor único, y regresión múltiple con todas las variables fisicoquímicas disponibles.

### 2. Hallazgos principales
El análisis reveló patrones relevantes: ninguna variable sigue distribución normal; *alcohol* y *sulphates* presentan relación positiva con la calidad, mientras que *volatile acidity* y *chlorides* muestran relación negativa significativa. Además, se identificó multicolinealidad entre *fixed acidity*, *citric acid* y *density*.

### 3. Respuesta a la pregunta de investigación
**"¿Qué variables químicas influyen más en la calidad del vino tinto y qué tan efectiva es una regresión lineal para predecir dicha calidad?"**

Los resultados del modelo múltiple muestran que *alcohol*, *sulphates* y *volatile acidity* son las variables más influyentes. El modelo de regresión lineal simple con `citric acid` explicó apenas el **~2–5%** de la variabilidad, mientras que el modelo múltiple alcanzó aproximadamente el **39% (R² ≈ 0.39)**, lo que sigue siendo limitado para predicción de alta precisión.

### 4. Comparativa de modelos
| Modelo | Predictores | R² (aprox.) | RMSE (aprox.) | MAE (aprox.) |
|--------|-------------|-------------|----------------|---------------|
| Simple | citric acid (1) | ~0.04 | ~0.15 | ~0.12 |
| Múltiple | Todas (11) | ~0.39 | ~0.12 | ~0.09 |

### 5. Limitaciones
- La variable *quality* es subjetiva y discreta, lo que dificulta la predicción mediante regresión.  
- El dataset no incluye información sensorial o de producción, lo cual reduce la capacidad explicativa.  
- La presencia de outliers y la ausencia de normalidad afectan modelos lineales tradicionales.  
- La regresión lineal no captura relaciones no lineales entre variables químicas.  
- Se detectó multicolinealidad en algunos predictores, lo que puede inflar los errores estándar de los coeficientes.

### 6. Recomendaciones
- Utilizar modelos no lineales como **Random Forest**, **Gradient Boosting** o **SVM**.  
- Probar enfoques de **clasificación multiclase** en lugar de regresión.  
- Aplicar técnicas de **balanceo de clases** para mejorar la predicción en calidades poco representadas.  
- Considerar **regularización Ridge o Lasso** para gestionar la multicolinealidad.  
- Incluir variables sensoriales y de proceso para mejorar el poder predictivo.